In [3]:
import json
import os
from pathlib import Path

def calculate_average_induction_scores(model_name):
    """
    Calculate the total average score across all induction tasks for a given model.
    
    Args:
        model_name (str): Name of the model (used in folder name)
    
    Returns:
        dict: Contains individual task scores and overall average
    """
    
    INDUCTION_TASKS = [
        'cause_and_effect', 'larger_animal', 'num_to_verbal','orthography_starts_with',
        'rhymes', 'synonyms', 'taxonomy_animal', 'translation_en-fr',
        'reverse_from_middle', 'smallest_item_length', 'smallest_even_no_sqrt', 'most_vowel_return_consonant',
        'detect_rhyme_and_rewrite', 'rank_by_protein','multi_lang_to_english','square_of_zodiac_animal',
        'alternate_synonym_antonym', 'most_consonant_return_vowel', 'least_unique_word_count', 'first_word_alphabetically_return_reverse'
    ]
    
    predictions_folder = Path(f"predictions_{model_name}")
    task_scores = {}
    missing_files = []
    
    print(f"Calculating scores for model: {model_name}")
    print(f"Looking in folder: {predictions_folder}")
    print("-" * 50)
    
    # Check if folder exists
    if not predictions_folder.exists():
        print(f"Error: Folder '{predictions_folder}' does not exist!")
        return None
    
    # Process each task
    for task in INDUCTION_TASKS:
        file_path = predictions_folder / f"{task}_with_scores.json"
        
        if file_path.exists():
            try:
                with open(file_path, 'r') as f:
                    data = json.load(f)
                
                # Extract weighted task score
                if 'weighted_task_score' in data:
                    score = data['weighted_task_score']
                    task_scores[task] = score
                    print(f"{task:<35}: {score:.4f}")
                else:
                    print(f"{task:<35}: Missing 'weighted_task_score' key")
                    missing_files.append(f"{task} (missing key)")
                    
            except json.JSONDecodeError:
                print(f"{task:<35}: Error reading JSON file")
                missing_files.append(f"{task} (JSON error)")
            except Exception as e:
                print(f"{task:<35}: Error - {str(e)}")
                missing_files.append(f"{task} (error)")
        else:
            print(f"{task:<35}: File not found")
            missing_files.append(task)
    
    print("-" * 50)
    
    # Calculate average
    if task_scores:
        total_average = sum(task_scores.values()) / len(task_scores)
        print(f"Total tasks processed: {len(task_scores)}/{len(INDUCTION_TASKS)}")
        print(f"TOTAL AVERAGE SCORE: {total_average:.4f}")
        
        if missing_files:
            print(f"\nMissing/Error files ({len(missing_files)}):")
            for missing in missing_files:
                print(f"  - {missing}")
    else:
        print("No valid scores found!")
        total_average = None
    
    return {
        'model_name': model_name,
        'task_scores': task_scores,
        'total_average': total_average,
        'tasks_processed': len(task_scores),
        'total_tasks': len(INDUCTION_TASKS),
        'missing_files': missing_files
    }

def compare_multiple_models(model_names):
    """
    Compare average scores across multiple models.
    
    Args:
        model_names (list): List of model names to compare
    """
    results = {}
    
    for model_name in model_names:
        print(f"\n{'='*60}")
        result = calculate_average_induction_scores(model_name)
        if result:
            results[model_name] = result
    
    # Summary comparison
    if len(results) > 1:
        print(f"\n{'='*60}")
        print("SUMMARY COMPARISON")
        print("="*60)
        
        for model_name, result in results.items():
            if result['total_average'] is not None:
                print(f"{model_name:<20}: {result['total_average']:.4f} ({result['tasks_processed']}/{result['total_tasks']} tasks)")
            else:
                print(f"{model_name:<20}: No valid scores")
    
    return results

# Example usage:
if __name__ == "__main__":
    # Replace with your actual model name
    #model_name = "openai_gpt-oss-20b"  # e.g., "qwq-32b", "gpt-4", etc.
    
    # For single model
    #result = calculate_average_induction_scores(model_name)
    
    # For multiple models comparison (uncomment if needed)
    model_names = ["nvidia_Nemotron-Research-Reasoning-Qwen-1.5B", 
                   "open-thoughts_OpenThinker-7B",
                   "openai_gpt-oss-20b",
                  "Qwen_QwQ-32B", "BytedTsinghua-SIA_DAPO-Qwen-32B"]
    results = compare_multiple_models(model_names)


Calculating scores for model: nvidia_Nemotron-Research-Reasoning-Qwen-1.5B
Looking in folder: predictions_nvidia_Nemotron-Research-Reasoning-Qwen-1.5B
--------------------------------------------------
cause_and_effect                   : 0.6014
larger_animal                      : 0.5327
num_to_verbal                      : 0.6113
orthography_starts_with            : 0.5429
rhymes                             : 0.5664
synonyms                           : 0.5551
taxonomy_animal                    : 0.6068
translation_en-fr                  : 0.6020
reverse_from_middle                : 0.5274
smallest_item_length               : 0.5497
smallest_even_no_sqrt              : 0.4618
most_vowel_return_consonant        : 0.5211
detect_rhyme_and_rewrite           : 0.5591
rank_by_protein                    : 0.5654
multi_lang_to_english              : 0.4824
square_of_zodiac_animal            : 0.5409
alternate_synonym_antonym          : 0.5474
most_consonant_return_vowel        : 0.4949
least